In [1]:
# Import Required Libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
import re
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Load and Prepare Data

In [2]:
# Load the enriched dataset
df = pd.read_csv('data/2026-01/turbo_az_2026-01.csv')

print(f"📊 Dataset loaded: {len(df):,} listings")
print(f"📋 Columns: {len(df.columns)}")
df.head()

📊 Dataset loaded: 34,887 listings
📋 Columns: 28


,url,title,brand,model,price,year,engine,mileage,location,datetime,date,time,scraped_at,detail_city,detail_brand,detail_model,detail_year,detail_body_type,detail_color,detail_engine,detail_mileage,detail_transmission,detail_drive_type,detail_new_used,detail_seats,detail_condition,detail_target_market,detail_owner_count
0,https://turbo.az/autos/9702812-abarth-500,Abarth 500,Abarth,NaN,21 500 ₼,2013,1.4 L,143 000 km,Bakı,"Bakı, dünən 15:08",2026-01-04,15:08,2026-01-05T13:51:48.481090,Bakı,Abarth,500,"2,013.00","Hetçbek, 3 qapı",Yaş Asfalt,1.4 L / 180 a.g. / Benzin,143 000 km,Mexaniki (MT),Ön,Xeyr,NaN,"Vuruğu yoxdur, rənglənməyib",NaN,0
1,https://turbo.az/autos/9959465-abarth-500,Abarth 500,Abarth,NaN,15 500 ₼,2013,1.4 L,130 000 km,Bakı,"Bakı, bugün 13:46",2026-01-05,13:46,2026-01-05T13:51:48.481250,Bakı,Abarth,500,"2,013.00","Hetçbek, 3 qapı",Ağ,1.4 L / 180 a.g. / Benzin,130 000 km,Mexaniki (MT),Ön,Xeyr,4,"Vuruğu yoxdur, rənglənməyib",Amerika,NaN
2,https://turbo.az/autos/10015096-abarth-500,Abarth 500,Abarth,NaN,16 000 ₼,2014,1.4 L,175 531 km,Bakı,"Bakı, 20.12.2025 13:28",2026-01-05,13:28,2026-01-05T13:51:48.481494,Bakı,Abarth,500,"2,014.00","Hetçbek, 3 qapı",Qara,1.4 L / 180 a.g. / Benzin,175 531 km,Mexaniki (MT),Ön,Xeyr,4,"Vuruğu yoxdur, rənglənməyib",Digər,NaN
3,https://turbo.az/autos/9997286-abarth-500,Abarth 500,Abarth,NaN,21 500 ₼,2016,1.4 L,172 150 km,Bakı,"Bakı, 12.12.2025 18:07",2026-01-05,18:07,2026-01-05T13:51:48.481590,Bakı,Abarth,500,"2,016.00","Hetçbek, 3 qapı",Qırmızı,1.4 L / 159 a.g. / Benzin,172 150 km,Avtomat (AT),Ön,Xeyr,4,"Vuruğu yoxdur, rənglənməyib",Amerika,NaN
4,https://turbo.az/autos/9222071-acura-mdx,Acura MDX,Acura,NaN,7 000 ₼,2000,3.5 L,164 690 km,Bakı,"Bakı, 29.12.2025 20:51",2026-01-05,20:51,2026-01-05T13:52:03.907757,Bakı,Acura,MDX,"2,000.00","Offroader / SUV, 5 qapı",Qara,3.5 L / 240 a.g. / Benzin,164 690 km,Avtomat (AT),Tam,Xeyr,NaN,"Vuruğu yoxdur, rənglənməyib",NaN,1


In [3]:
# Data cleaning and preparation

# Clean price column - extract numeric value
df['price_numeric'] = df['price'].str.replace(r'[\s₼]', '', regex=True)
df['price_numeric'] = pd.to_numeric(df['price_numeric'], errors='coerce')

# Clean mileage - extract numeric value from detail_mileage
df['mileage_numeric'] = df['detail_mileage'].str.replace(r'[\s,km]', '', regex=True)
df['mileage_numeric'] = pd.to_numeric(df['mileage_numeric'], errors='coerce')

# Extract listing date from datetime field
# Format options: "Bakı, 02.01.2026 13:38" (absolute) or "Bakı, bugün 13:46" (today) or "Bakı, dünən 15:08" (yesterday)
def parse_listing_date(datetime_str):
    """Parse datetime handling absolute dates, 'bugün' (today), and 'dünən' (yesterday)"""
    if pd.isna(datetime_str):
        return None
    
    dt_str = str(datetime_str).strip()
    
    # Try to extract absolute date first (DD.MM.YYYY format)
    date_match = re.search(r'(\d{2}\.\d{2}\.\d{4})', dt_str)
    if date_match:
        try:
            return pd.to_datetime(date_match.group(1), format='%d.%m.%Y')
        except Exception as e:
            print(f"Date parsing error: {e} for string: {dt_str}")
            pass
    
    # Handle relative dates (Azerbaijani)
    if 'bugün' in dt_str:  # "today"
        return pd.Timestamp.today().normalize()
    elif 'dünən' in dt_str:  # "yesterday"
        return (pd.Timestamp.today() - pd.Timedelta(days=1)).normalize()
    
    return None

df['listing_date'] = df['datetime'].apply(parse_listing_date)

# Year as integer
df['year_numeric'] = pd.to_numeric(df['year'], errors='coerce')

# Calculate car age
df['car_age'] = 2026 - df['year_numeric']

# Extract horsepower from detail_engine (e.g., "2.0 L / 180 a.g. / Benzin")
df['horsepower'] = df['detail_engine'].str.extract(r'(\d+)\s*a\.g\.', expand=False)
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')

# Extract fuel type
df['fuel_type'] = df['detail_engine'].str.extract(r'/\s*([A-Za-zə]+)$', expand=False)

# Extract engine displacement
df['engine_displacement'] = df['detail_engine'].str.extract(r'^([\d\.]+)\s*L', expand=False)
df['engine_displacement'] = pd.to_numeric(df['engine_displacement'], errors='coerce')

print("✅ Data cleaning complete!")
print(f"\n📈 Price range: {df['price_numeric'].min():,.0f} ₼ - {df['price_numeric'].max():,.0f} ₼")
print(f"📅 Year range: {df['year_numeric'].min():.0f} - {df['year_numeric'].max():.0f}")
print(f"🛣️ Mileage range: {df['mileage_numeric'].min():,.0f} - {df['mileage_numeric'].max():,.0f} km")
print(f"📅 Listing dates extracted: {df['listing_date'].notna().sum():,} ({df['listing_date'].notna().sum()/len(df)*100:.1f}%)")

✅ Data cleaning complete!

📈 Price range: 500 ₼ - 8,211,000 ₼
📅 Year range: 1939 - 2025
🛣️ Mileage range: 0 - 3,700,000 km
📅 Listing dates extracted: 34,887 (100.0%)


In [4]:
df['listing_date'].min(), df['listing_date'].max()

(Timestamp('2025-12-06 00:00:00'), Timestamp('2026-01-12 00:00:00'))

In [5]:
# Dataset overview
print("📋 COLUMN INFORMATION")
print("="*60)
for col in df.columns:
    non_null = df[col].notna().sum()
    pct = (non_null / len(df)) * 100
    print(f"{col:35} {non_null:6,} ({pct:5.1f}%)")

📋 COLUMN INFORMATION
url                                 34,887 (100.0%)
title                               34,887 (100.0%)
brand                               34,887 (100.0%)
model                               11,211 ( 32.1%)
price                               34,887 (100.0%)
year                                34,887 (100.0%)
engine                              34,887 (100.0%)
mileage                             34,410 ( 98.6%)
location                            34,887 (100.0%)
datetime                            34,887 (100.0%)
date                                34,887 (100.0%)
time                                34,887 (100.0%)
scraped_at                          34,887 (100.0%)
detail_city                         34,747 ( 99.6%)
detail_brand                        34,747 ( 99.6%)
detail_model                        34,747 ( 99.6%)
detail_year                         34,747 ( 99.6%)
detail_body_type                    34,747 ( 99.6%)
detail_color                        34,747 

## 2. Market Overview

In [6]:
# Key metrics
total_listings = len(df)
unique_brands = df['detail_brand'].nunique()
unique_locations = df['location'].nunique()
avg_price = df['price_numeric'].mean()
median_price = df['price_numeric'].median()
avg_mileage = df['mileage_numeric'].mean()
avg_year = df['year_numeric'].mean()

print("🚗 TURBO.AZ MARKET OVERVIEW")
print("="*50)
print(f"📊 Total Listings:     {total_listings:,}")
print(f"🏷️  Unique Brands:      {unique_brands}")
print(f"📍 Unique Locations:   {unique_locations}")
print(f"💰 Average Price:      {avg_price:,.0f} ₼")
print(f"💵 Median Price:       {median_price:,.0f} ₼")
print(f"🛣️  Average Mileage:    {avg_mileage:,.0f} km")
print(f"📅 Average Year:       {avg_year:.1f}")

🚗 TURBO.AZ MARKET OVERVIEW
📊 Total Listings:     34,887
🏷️  Unique Brands:      210
📍 Unique Locations:   68
💰 Average Price:      30,194 ₼
💵 Median Price:       18,500 ₼
🛣️  Average Mileage:    202,796 km
📅 Average Year:       2011.0


## 3. Brand Analysis

In [7]:
# Top 20 brands by listing count
top_brands = df['detail_brand'].value_counts().head(20).reset_index()
top_brands.columns = ['Brand', 'Count']
top_brands['Percentage'] = (top_brands['Count'] / len(df) * 100).round(1)

fig = px.bar(
    top_brands,
    x='Count',
    y='Brand',
    orientation='h',
    title='🏷️ Top 20 Car Brands by Number of Listings',
    text='Count',
    color='Count',
    color_continuous_scale='Blues'
)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.show()

In [8]:
# Brand market share (pie chart)
top_10_brands = df['detail_brand'].value_counts().head(10)
other_count = len(df) - top_10_brands.sum()

brand_share = pd.DataFrame({
    'Brand': list(top_10_brands.index) + ['Others'],
    'Count': list(top_10_brands.values) + [other_count]
})

fig = px.pie(
    brand_share,
    values='Count',
    names='Brand',
    title='🥧 Market Share by Brand (Top 10 + Others)',
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=500)
fig.show()

In [9]:
# Average price by brand (top 15 brands by volume)
top_15_brand_names = df['detail_brand'].value_counts().head(15).index
brand_price = df[df['detail_brand'].isin(top_15_brand_names)].groupby('detail_brand').agg({
    'price_numeric': ['mean', 'median', 'count']
}).round(0)
brand_price.columns = ['Mean Price', 'Median Price', 'Count']
brand_price = brand_price.sort_values('Median Price', ascending=True).reset_index()

fig = px.bar(
    brand_price,
    x='Median Price',
    y='detail_brand',
    orientation='h',
    title='💰 Median Price by Brand (Top 15 Brands by Volume)',
    text='Median Price',
    color='Median Price',
    color_continuous_scale='Greens'
)
fig.update_traces(texttemplate='%{text:,.0f} ₼', textposition='outside')
fig.update_layout(height=500, xaxis_title='Median Price (₼)', yaxis_title='Brand')
fig.show()

## 4. Price Analysis

In [10]:
# Price distribution (filtering outliers for better visualization)
price_filtered = df[df["price_numeric"] <= 100000]["price_numeric"].dropna()
median_val = price_filtered.median()
mean_val = price_filtered.mean()

fig = px.histogram(
    price_filtered,
    nbins=50,
    title="💰 Price Distribution (≤100,000 ₼)",
    labels={"value": "Price (₼)", "count": "Number of Listings"},
)

fig.add_vline(
    x=median_val,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Median: {median_val:,.0f} ₼",
    annotation_position="top left"
)

fig.add_vline(
    x=mean_val,
    line_dash="dash",
    line_color="green",
    annotation_text=f"Mean: {mean_val:,.0f} ₼",
    annotation_position="top right"
)

fig.update_layout(height=400)
fig.show()

In [11]:
# Price segments
def price_segment(price):
    if pd.isna(price):
        return 'Unknown'
    elif price < 5000:
        return '< 5,000 ₼'
    elif price < 10000:
        return '5,000 - 10,000 ₼'
    elif price < 20000:
        return '10,000 - 20,000 ₼'
    elif price < 30000:
        return '20,000 - 30,000 ₼'
    elif price < 50000:
        return '30,000 - 50,000 ₼'
    elif price < 100000:
        return '50,000 - 100,000 ₼'
    else:
        return '> 100,000 ₼'

df['price_segment'] = df['price_numeric'].apply(price_segment)

segment_order = ['< 5,000 ₼', '5,000 - 10,000 ₼', '10,000 - 20,000 ₼', 
                 '20,000 - 30,000 ₼', '30,000 - 50,000 ₼', '50,000 - 100,000 ₼', '> 100,000 ₼']

segment_counts = df['price_segment'].value_counts().reindex(segment_order)

fig = px.bar(
    x=segment_counts.index,
    y=segment_counts.values,
    title='📊 Listings by Price Segment',
    labels={'x': 'Price Segment', 'y': 'Number of Listings'},
    color=segment_counts.values,
    color_continuous_scale='Viridis'
)
fig.update_layout(height=400)
fig.show()

In [12]:
# Price statistics table
price_stats = df['price_numeric'].describe().round(0)
print("💰 PRICE STATISTICS")
print("="*40)
print(f"Count:          {price_stats['count']:,.0f}")
print(f"Mean:           {price_stats['mean']:,.0f} ₼")
print(f"Std Deviation:  {price_stats['std']:,.0f} ₼")
print(f"Min:            {price_stats['min']:,.0f} ₼")
print(f"25th Percentile:{price_stats['25%']:,.0f} ₼")
print(f"Median (50%):   {price_stats['50%']:,.0f} ₼")
print(f"75th Percentile:{price_stats['75%']:,.0f} ₼")
print(f"Max:            {price_stats['max']:,.0f} ₼")

💰 PRICE STATISTICS
Count:          34,887
Mean:           30,194 ₼
Std Deviation:  75,273 ₼
Min:            500 ₼
25th Percentile:9,800 ₼
Median (50%):   18,500 ₼
75th Percentile:33,500 ₼
Max:            8,211,000 ₼


## 5. Year & Age Analysis

In [13]:
# Distribution by year
year_counts = df['year_numeric'].value_counts().sort_index()
year_counts = year_counts[year_counts.index >= 1990]  # Filter from 1990

fig = px.bar(
    x=year_counts.index,
    y=year_counts.values,
    title='📅 Listings by Manufacturing Year (1990+)',
    labels={'x': 'Year', 'y': 'Number of Listings'},
    color=year_counts.values,
    color_continuous_scale='Plasma'
)
fig.update_layout(height=400)
fig.show()

In [14]:
# Price vs Year scatter
sample_df = df[(df['price_numeric'] <= 100000) & (df['year_numeric'] >= 2000)].sample(min(5000, len(df)))

fig = px.scatter(
    sample_df,
    x='year_numeric',
    y='price_numeric',
    color='detail_brand',
    title='💰 Price vs Year (Sample of listings, ≤100k ₼, 2000+)',
    labels={'year_numeric': 'Year', 'price_numeric': 'Price (₼)'},
    opacity=0.6
)
fig.update_layout(height=500)
fig.show()

In [15]:
# Average price by year
price_by_year = df[df['year_numeric'] >= 2000].groupby('year_numeric')['price_numeric'].agg(['mean', 'median', 'count'])
price_by_year = price_by_year[price_by_year['count'] >= 20]  # At least 20 listings

fig = go.Figure()
fig.add_trace(go.Scatter(x=price_by_year.index, y=price_by_year['mean'], name='Mean Price', mode='lines+markers'))
fig.add_trace(go.Scatter(x=price_by_year.index, y=price_by_year['median'], name='Median Price', mode='lines+markers'))

fig.update_layout(
    title='📈 Average & Median Price by Year (2000+, min 20 listings)',
    xaxis_title='Year',
    yaxis_title='Price (₼)',
    height=400
)
fig.show()

## 6. Location Analysis

In [16]:
# Top 20 locations
top_locations = df['location'].value_counts().head(20).reset_index()
top_locations.columns = ['Location', 'Count']
top_locations['Percentage'] = (top_locations['Count'] / len(df) * 100).round(1)

fig = px.bar(
    top_locations,
    x='Count',
    y='Location',
    orientation='h',
    title='📍 Top 20 Locations by Number of Listings',
    text='Count',
    color='Count',
    color_continuous_scale='Oranges'
)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.show()

In [17]:
# Average price by location (top 15)
top_15_locations = df['location'].value_counts().head(15).index
price_by_location = df[df['location'].isin(top_15_locations)].groupby('location')['price_numeric'].agg(['mean', 'median', 'count']).round(0)
price_by_location.columns = ['Mean Price', 'Median Price', 'Count']
price_by_location = price_by_location.sort_values('Median Price', ascending=True).reset_index()

fig = px.bar(
    price_by_location,
    x='Median Price',
    y='location',
    orientation='h',
    title='💰 Median Price by Location (Top 15 Locations)',
    text='Median Price',
    color='Median Price',
    color_continuous_scale='RdYlGn'
)
fig.update_traces(texttemplate='%{text:,.0f} ₼', textposition='outside')
fig.update_layout(height=500, xaxis_title='Median Price (₼)', yaxis_title='')
fig.show()

## 7. Body Type Analysis

In [18]:
# Body type distribution
body_types = df['detail_body_type'].value_counts().head(15).reset_index()
body_types.columns = ['Body Type', 'Count']

fig = px.bar(
    body_types,
    x='Count',
    y='Body Type',
    orientation='h',
    title='🚙 Top 15 Body Types',
    text='Count',
    color='Count',
    color_continuous_scale='Purples'
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.show()

In [19]:
# Price by body type
body_price = df.groupby('detail_body_type')['price_numeric'].agg(['median', 'count'])
body_price = body_price[body_price['count'] >= 50].sort_values('median', ascending=True).reset_index()
body_price.columns = ['Body Type', 'Median Price', 'Count']

fig = px.bar(
    body_price,
    x='Median Price',
    y='Body Type',
    orientation='h',
    title='💰 Median Price by Body Type (min 50 listings)',
    text='Median Price',
    color='Median Price',
    color_continuous_scale='Teal'
)
fig.update_traces(texttemplate='%{text:,.0f} ₼', textposition='outside')
fig.update_layout(height=500)
fig.show()

## 8. Transmission & Drivetrain Analysis

In [20]:
# Transmission distribution
transmission_counts = df['detail_transmission'].value_counts().reset_index()
transmission_counts.columns = ['Transmission', 'Count']

fig = px.pie(
    transmission_counts,
    values='Count',
    names='Transmission',
    title='⚙️ Transmission Type Distribution',
    hole=0.3
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=450)
fig.show()

In [21]:
# Drive type distribution
drive_counts = df['detail_drive_type'].value_counts().reset_index()
drive_counts.columns = ['Drive Type', 'Count']

fig = px.pie(
    drive_counts,
    values='Count',
    names='Drive Type',
    title='🚗 Drive Type Distribution',
    hole=0.3
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=400)
fig.show()

## 9. Fuel Type & Engine Analysis

In [22]:
# Fuel type distribution
fuel_counts = df['fuel_type'].value_counts().reset_index()
fuel_counts.columns = ['Fuel Type', 'Count']

fig = px.bar(
    fuel_counts,
    x='Fuel Type',
    y='Count',
    title='⛽ Fuel Type Distribution',
    color='Count',
    color_continuous_scale='Reds'
)
fig.update_layout(height=400)
fig.show()

In [23]:
# Price by fuel type
fuel_price = df.groupby('fuel_type')['price_numeric'].agg(['median', 'mean', 'count'])
fuel_price = fuel_price[fuel_price['count'] >= 50].sort_values('median', ascending=False).reset_index()
fuel_price.columns = ['Fuel Type', 'Median Price', 'Mean Price', 'Count']

fig = px.bar(
    fuel_price,
    x='Fuel Type',
    y='Median Price',
    title='💰 Median Price by Fuel Type',
    text='Median Price',
    color='Median Price',
    color_continuous_scale='YlOrRd'
)
fig.update_traces(texttemplate='%{text:,.0f} ₼', textposition='outside')
fig.update_layout(height=400)
fig.show()

In [24]:
# Engine displacement distribution
engine_disp = df['engine_displacement'].value_counts().head(15).sort_index().reset_index()
engine_disp.columns = ['Engine (L)', 'Count']

fig = px.bar(
    engine_disp.sort_values('Count', ascending=False),
    x='Engine (L)',
    y='Count',
    title='🔧 Engine Displacement Distribution (Top 15)',
    color='Count',
    color_continuous_scale='Blues'
)
fig.update_layout(height=400)
fig.show()

## 10. Target Market Analysis

In [25]:
# Target market distribution
market_counts = df['detail_target_market'].value_counts().reset_index()
market_counts.columns = ['Target Market', 'Count']
market_counts['Percentage'] = (market_counts['Count'] / market_counts['Count'].sum() * 100).round(1)

fig = px.bar(
    market_counts,
    x='Count',
    y='Target Market',
    orientation='h',
    title='🌍 Cars by Target Market / Origin',
    text='Count',
    color='Count',
    color_continuous_scale='Viridis'
)
fig.update_layout(height=450, yaxis={'categoryorder': 'total ascending'})
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.show()

In [26]:
# Price by target market
market_price = df.groupby('detail_target_market')['price_numeric'].agg(['median', 'count'])
market_price = market_price[market_price['count'] >= 50].sort_values('median', ascending=True).reset_index()
market_price.columns = ['Target Market', 'Median Price', 'Count']

fig = px.bar(
    market_price,
    x='Median Price',
    y='Target Market',
    orientation='h',
    title='💰 Median Price by Target Market',
    text='Median Price',
    color='Median Price',
    color_continuous_scale='Sunset'
)
fig.update_traces(texttemplate='%{text:,.0f} ₼', textposition='outside')
fig.update_layout(height=400)
fig.show()

## 11. Vehicle Condition Analysis

In [27]:
# Condition distribution
condition_counts = df['detail_condition'].value_counts().reset_index()
condition_counts.columns = ['Condition', 'Count']
condition_counts['Percentage'] = (condition_counts['Count'] / condition_counts['Count'].sum() * 100).round(1)

fig = px.bar(
    condition_counts,
    x='Count',
    y='Condition',
    orientation='h',
    title='🔍 Vehicle Condition Distribution',
    text=condition_counts.apply(lambda x: f"{x['Count']:,} ({x['Percentage']}%)", axis=1),
    color='Count',
    color_continuous_scale='RdYlGn'
)
fig.update_layout(height=350, yaxis={'categoryorder': 'total ascending'})
fig.show()

In [28]:
# Price by condition
condition_price = df.groupby('detail_condition')['price_numeric'].agg(['median', 'mean', 'count'])
condition_price = condition_price.sort_values('median', ascending=False).reset_index()
condition_price.columns = ['Condition', 'Median Price', 'Mean Price', 'Count']

print("💰 PRICE BY VEHICLE CONDITION")
print("="*70)
for _, row in condition_price.iterrows():
    print(f"{row['Condition']:45} Median: {row['Median Price']:10,.0f} ₼  (n={row['Count']:,})")

💰 PRICE BY VEHICLE CONDITION
Vuruğu yoxdur, rənglənməyib                   Median:     20,300 ₼  (n=30,292)
Vuruğu yoxdur, rənglənib                      Median:     10,900 ₼  (n=3,656)
Vuruğu var, rənglənməyib                      Median:      9,650 ₼  (n=410)
Vuruğu var, rənglənib                         Median:      9,000 ₼  (n=388)


## 12. Color Analysis

In [29]:
# Color distribution
color_counts = df['detail_color'].value_counts().head(15).reset_index()
color_counts.columns = ['Color', 'Count']

# Map colors to actual colors for visualization
color_map = {
    'Qara': '#1a1a1a', 'Ağ': '#f5f5f5', 'Gümüşü': '#c0c0c0', 'Göy': '#0066cc',
    'Boz': '#808080', 'Yaş Asfalt': '#4a4a4a', 'Qırmızı': '#cc0000', 'Yaşıl': '#228b22',
    'Qəhvəyi': '#8b4513', 'Bej': '#f5f5dc', 'Narıncı': '#ff8c00', 'Sarı': '#ffd700',
    'Tünd yaşıl': '#006400', 'Tünd göy': '#00008b', 'Tünd qırmızı': '#8b0000'
}

fig = px.bar(
    color_counts,
    x='Count',
    y='Color',
    orientation='h',
    title='🎨 Top 15 Car Colors',
    text='Count',
    color='Color',
    color_discrete_map=color_map
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'}, showlegend=False)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.show()

## 13. Mileage Analysis

In [30]:
# Mileage distribution (filtering extremes)
mileage_filtered = df[(df['mileage_numeric'] > 0) & (df['mileage_numeric'] <= 500000)]['mileage_numeric']

fig = px.histogram(
    mileage_filtered,
    nbins=50,
    title='🛣️ Mileage Distribution (1 - 500,000 km)',
    labels={'value': 'Mileage (km)', 'count': 'Number of Listings'}
)
fig.add_vline(x=mileage_filtered.median(), line_dash="dash", line_color="red",
              annotation_text=f"Median: {mileage_filtered.median():,.0f} km")
fig.update_layout(height=400)
fig.show()

In [31]:
# Mileage segments
def mileage_segment(km):
    if pd.isna(km) or km <= 0:
        return 'New/Unknown'
    elif km < 50000:
        return '< 50k km'
    elif km < 100000:
        return '50k - 100k km'
    elif km < 150000:
        return '100k - 150k km'
    elif km < 200000:
        return '150k - 200k km'
    elif km < 300000:
        return '200k - 300k km'
    else:
        return '> 300k km'

df['mileage_segment'] = df['mileage_numeric'].apply(mileage_segment)

segment_order = ['New/Unknown', '< 50k km', '50k - 100k km', '100k - 150k km', 
                 '150k - 200k km', '200k - 300k km', '> 300k km']
mileage_segments = df['mileage_segment'].value_counts().reindex(segment_order)

fig = px.bar(
    x=mileage_segments.index,
    y=mileage_segments.values,
    title='📊 Listings by Mileage Segment',
    labels={'x': 'Mileage Segment', 'y': 'Number of Listings'},
    color=mileage_segments.values,
    color_continuous_scale='Bluered'
)
fig.update_layout(height=400)
fig.show()

In [32]:
# Average mileage by year
mileage_by_year = df[(df['year_numeric'] >= 2000) & (df['mileage_numeric'] > 0)].groupby('year_numeric')['mileage_numeric'].agg(['mean', 'median', 'count'])
mileage_by_year = mileage_by_year[mileage_by_year['count'] >= 20]

fig = go.Figure()
fig.add_trace(go.Scatter(x=mileage_by_year.index, y=mileage_by_year['mean'], name='Mean Mileage', mode='lines+markers'))
fig.add_trace(go.Scatter(x=mileage_by_year.index, y=mileage_by_year['median'], name='Median Mileage', mode='lines+markers'))

fig.update_layout(
    title='📈 Average Mileage by Manufacturing Year (2000+)',
    xaxis_title='Year',
    yaxis_title='Mileage (km)',
    height=400
)
fig.show()

## 14. Advanced Analytics

In [35]:
# Price vs Mileage correlation
sample = df[(df['price_numeric'] <= 80000) & (df['mileage_numeric'] > 0) & (df['mileage_numeric'] <= 400000)].sample(min(3000, len(df)))

fig = px.scatter(
    sample,
    x='mileage_numeric',
    y='price_numeric',
    color='car_age',
    title='💰 Price vs Mileage (colored by car age)',
    labels={'mileage_numeric': 'Mileage (km)', 'price_numeric': 'Price (₼)', 'car_age': 'Car Age'},
    opacity=0.5,
    trendline='ols'
)
fig.update_layout(height=500)
fig.show()

In [36]:
# Correlation heatmap
numeric_cols = ['price_numeric', 'year_numeric', 'mileage_numeric', 'car_age', 'horsepower', 'engine_displacement']
corr_matrix = df[numeric_cols].corr()

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    title='📊 Correlation Matrix',
    color_continuous_scale='RdBu_r'
)
fig.update_layout(height=500)
fig.show()

In [37]:
# Top models within brands
print("🏆 TOP 5 MODELS FOR EACH TOP 10 BRAND")
print("="*60)

for brand in top_10_brands.index:
    brand_models = df[df['detail_brand'] == brand]['detail_model'].value_counts().head(5)
    print(f"\n{brand}:")
    for model, count in brand_models.items():
        print(f"  - {model}: {count:,}")

🏆 TOP 5 MODELS FOR EACH TOP 10 BRAND

Mercedes:
  - E 220: 317
  - C 180: 299
  - 190: 255
  - E 240: 223
  - C 200: 208

LADA (VAZ):
  - 2107: 1,207
  - 2106: 702
  - Niva: 393
  - 2115: 274
  - 21099: 175

Hyundai:
  - Sonata: 623
  - Elantra: 528
  - Santa Fe: 441
  - Accent: 251
  - Tucson: 177

BMW:
  - X5: 326
  - 528: 287
  - 530: 182
  - 328: 129
  - 520: 126

Toyota:
  - Prius: 401
  - Land Cruiser Prado: 335
  - Camry: 305
  - Corolla: 283
  - Land Cruiser: 197

Kia:
  - Sorento: 470
  - Rio: 292
  - Optima: 257
  - Sportage: 219
  - K5: 156

Changan:
  - Uni-Z: 528
  - Qiyuan Q07: 427
  - Qiyuan A05: 170
  - Qiyuan Q05: 142
  - CS 55 Plus: 108

Ford:
  - Transit: 860
  - Fusion (North America): 410
  - Focus: 63
  - Mustang: 44
  - Escape: 29

Chevrolet:
  - Cruze: 643
  - Malibu: 191
  - Cobalt: 153
  - Aveo: 123
  - Equinox: 110

Opel:
  - Astra: 789
  - Vectra: 247
  - Zafira: 82
  - Corsa: 35
  - Omega: 29


In [38]:
# Brand + Model heatmap (top combinations)
top_5_brands = df['detail_brand'].value_counts().head(5).index
brand_year = df[df['detail_brand'].isin(top_5_brands)].groupby(['detail_brand', 'year_numeric']).size().reset_index(name='count')
brand_year = brand_year[(brand_year['year_numeric'] >= 2010) & (brand_year['year_numeric'] <= 2025)]
brand_year_pivot = brand_year.pivot(index='detail_brand', columns='year_numeric', values='count').fillna(0)

fig = px.imshow(
    brand_year_pivot,
    text_auto='.0f',
    title='🔥 Heatmap: Top 5 Brands by Year (2010-2025)',
    color_continuous_scale='YlOrRd',
    aspect='auto'
)
fig.update_layout(height=350)
fig.show()

## 15. Summary Statistics

In [39]:
# Final summary
print("="*70)
print("📋 TURBO.AZ DATASET SUMMARY")
print("="*70)
print("\n📊 DATASET SIZE")
print(f"   Total listings: {total_listings:,}")
print(f"   Unique brands: {df['detail_brand'].nunique()}")
print(f"   Unique models: {df['detail_model'].nunique()}")
print(f"   Unique locations: {df['location'].nunique()}")

print("\n💰 PRICE")
print(f"   Mean: {df['price_numeric'].mean():,.0f} ₼")
print(f"   Median: {df['price_numeric'].median():,.0f} ₼")
print(f"   Range: {df['price_numeric'].min():,.0f} - {df['price_numeric'].max():,.0f} ₼")

print("\n📅 YEAR")
print(f"   Mean: {df['year_numeric'].mean():.1f}")
print(f"   Range: {df['year_numeric'].min():.0f} - {df['year_numeric'].max():.0f}")

print("\n🛣️ MILEAGE (for cars with mileage > 0)")
valid_mileage = df[df['mileage_numeric'] > 0]['mileage_numeric']
print(f"   Mean: {valid_mileage.mean():,.0f} km")
print(f"   Median: {valid_mileage.median():,.0f} km")

print("\n🏆 TOP 5 BRANDS")
for brand, count in df['detail_brand'].value_counts().head(5).items():
    pct = count/len(df)*100
    print(f"   {brand}: {count:,} ({pct:.1f}%)")

print("\n📍 TOP 5 LOCATIONS")
for loc, count in df['location'].value_counts().head(5).items():
    pct = count/len(df)*100
    print(f"   {loc}: {count:,} ({pct:.1f}%)")

print("\n✅ ENRICHMENT QUALITY")
enriched = df['detail_brand'].notna().sum()
print(f"   Successfully enriched: {enriched:,} ({enriched/len(df)*100:.1f}%)")
print("="*70)

📋 TURBO.AZ DATASET SUMMARY

📊 DATASET SIZE
   Total listings: 34,887
   Unique brands: 210
   Unique models: 2075
   Unique locations: 68

💰 PRICE
   Mean: 30,194 ₼
   Median: 18,500 ₼
   Range: 500 - 8,211,000 ₼

📅 YEAR
   Mean: 2011.0
   Range: 1939 - 2025

🛣️ MILEAGE (for cars with mileage > 0)
   Mean: 226,004 km
   Median: 205,000 km

🏆 TOP 5 BRANDS
   Mercedes: 5,218 (15.0%)
   LADA (VAZ): 3,647 (10.5%)
   Hyundai: 2,908 (8.3%)
   BMW: 2,216 (6.4%)
   Toyota: 2,178 (6.2%)

📍 TOP 5 LOCATIONS
   Bakı: 23,269 (66.7%)
   Sumqayıt: 2,249 (6.4%)
   Gəncə: 1,610 (4.6%)
   Lənkəran: 385 (1.1%)
   Şəmkir: 345 (1.0%)

✅ ENRICHMENT QUALITY
   Successfully enriched: 34,747 (99.6%)


In [40]:
# Top 10 Most Expensive Cars
expensive_cars = df.sort_values('price_numeric', ascending=False).head(10)[['title', 'price', 'year', 'mileage', 'location']]
print("💎 TOP 10 MOST EXPENSIVE CARS")
print("="*80)
display(expensive_cars)

# Oldest Cars
oldest_cars = df.sort_values('year_numeric', ascending=True).head(10)[['title', 'price', 'year', 'location']]
print("\n👴 OLDEST CARS IN THE MARKET")
print("="*80)
display(oldest_cars)

# Highest Mileage Cars
high_mileage = df.sort_values('mileage_numeric', ascending=False).head(10)[['title', 'price', 'year', 'detail_mileage', 'location']]
print("\n🛣️ CARS WITH HIGHEST MILEAGE")
print("="*80)
display(high_mileage)

💎 TOP 10 MOST EXPENSIVE CARS


,title,price,year,mileage,location
25180,Mercedes AMG One,8 211 000 ₼,2025,75 km,Bakı
2691,Bugatti Chiron Pur Sport,7 555 555 ₼,2023,9 334 km,Bakı
30854,Rolls-Royce Phantom,2 077 777 ₼,2025,0 km,Bakı
26424,Mercedes G 63 AMG,1 444 444 ₼,2025,0 km,Bakı
25309,Mercedes G 63 AMG,1 164 500 ₼,2025,0 km,Bakı
30856,Rolls-Royce Spectre,952 000 ₼,2025,NaN,Bakı
30857,Rolls-Royce Spectre,952 000 ₼,2025,NaN,Bakı
385,Bentley Continental GTC,873 796 ₼,2025,0 km,Bakı
399,Bentley Continental GT,853 937 ₼,2025,0 km,Bakı
401,Bentley Continental GT,834 078 ₼,2025,0 km,Bakı



👴 OLDEST CARS IN THE MARKET


,title,price,year,location
9362,GAZ M-1,161 500 ₼,1939,Bakı
9452,GAZ 67,45 000 ₼,1947,Bakı
9505,GAZ 69,9 500 ₼,1950,Bakı
9545,"GAZ M-20 ""Pobeda""",254 830 ₼,1952,Bakı
9108,GAZ 12 ZIM,68 000 ₼,1954,Bakı
27978,Moskvich 401,42 500 ₼,1955,Bakı
9173,GAZ 69,12 500 ₼,1956,Şəmkir
9475,"GAZ M-20 ""Pobeda""",5 000 ₼,1958,Bakı
27979,Moskvich 407,4 700 ₼,1958,Bakı
9110,GAZ M-21,2 400 ₼,1959,Xaçmaz



🛣️ CARS WITH HIGHEST MILEAGE


,title,price,year,detail_mileage,location
31079,Shacman F2000,35 000 ₼,2011,3 700 000 km,Bakı
21833,MAN TGA 18.440,28 500 ₼,2007,3 000 000 km,Astara
30727,Renault Premium 460,30 000 ₼,2011,2 554 245 km,Astara
14143,KamAz 53212,17 500 ₼,1988,2 000 000 km,Bakı
30992,Scania R420,50 000 ₼,2008,2 000 000 km,Bakı
6955,DAF 105 XF,31 000 ₼,2008,1 700 000 km,Bakı
30976,Scania R420,51 500 ₼,2009,1 603 257 km,Gəncə
25669,Mercedes Actros 1843,28 500 ₼,2000,1 555 000 km,Bakı
26145,Mercedes Actros 2544,42 000 ₼,2000,1 500 000 km,Bakı
6987,DAF 105 XF,24 500 ₼,2008,1 500 000 km,Bakı
